# 6. Streamlit app for classification model

🔹 A. Exploration (historical data) - transparency + insight

Users can:

- Select state
- Select time range  
- View:
    - biodiversity observations
    - anomalies (your target)
    - weather variables

🔹 B. Prediction (model) - real-world application

Users can:

- Input:
    - state
    - month/year (or future scenario)
    - weather conditions (different climate change scenarios)
- Output:
    - probability of biodiversity anomaly
    - classification (shock / no shock)

In [29]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib

In [40]:
# Load data
df = pd.read_parquet('../Data/Processed/full_df.parquet')

# Load model
model_bundle = joblib.load("../models/production_model.pkl")
model = model_bundle['model']

# Get features
features = model_bundle['features']

Define scenarios  

Key Climate Change Scenarios (IPCC 6th Assessment Report):

    SSP1-1.9 (Very Low Emissions): A best-case scenario aiming to keep global warming below
    by 2100 with rapid decarbonization and high global cooperation.  
    SSP1-2.6 (Low Emissions): A scenario where emissions decline after 2050, resulting in intermediate climate goals.  
    SSP2-4.5 (Intermediate Emissions): A "middle-of-the-road" scenario where emissions fluctuate and flatten by mid-century, leading to moderate changes.  
    SSP3-7.0 (High Emissions): A future with low international cooperation, slow technological growth, and high emissions.  
    SSP5-8.5 (Very High Emissions): A worst-case scenario featuring high fossil fuel dependence, leading to rapid temperature increases (by 2100) and severe impacts. 


Temperature Scenarios (2021-2100 Projections)

    Low Emissions (SSP1-1.9/2.6): Aims to keep warming closer to
    to 1.5°C - 2°C
    High Emissions (SSP5-8.5): Temperatures could rise by 3.3°C to 5.7°C, with significant Arctic amplification.
    Regional Variation: Land areas warm faster than oceans, and the Arctic is warming significantly faster than the global average. 

Precipitation Scenarios

    Seasonal Changes: Winters may see up to 20% more rain in some regions, while summers could face up to 25% less, exacerbating water shortages.
    Increased Intensity: Extreme precipitation events are expected to increase in frequency and intensity, causing higher flood risks. 

Need a slider for:  
    - temp increase (translate degrees C to z-values) (let user choose scenario e.g. 2C warming by 2100, but reduce increase for 10 years)  
    - (with temp increase, also increase likelihood of extreme hot days?)  
    - precipitation change (increase winter rain and decrease summer rain by same amount from baseline)  
    - (with precipitation change also increase frequency of extreme precipitation events)  

Also give preset scenarios based on ICPP projections

In [ ]:
SCENARIOS = {
    "Moderate warming": {
        "temp_start": 0.5,
        "temp_trend": 0.03,
        "precip_start": 0.0,
        "precip_trend": -0.01
    },
    "Severe warming & drying": {
        "temp_start": 1.0,
        "temp_trend": 0.08,
        "precip_start": -0.5,
        "precip_trend": -0.04
    },
    "Wet + unstable": {
        "temp_start": 0.7,
        "temp_trend": 0.04,
        "precip_start": 0.5,
        "precip_trend": 0.02
    }
}

Define function to simulate future data

In [ ]:
def simulate_future(
    state,
    start_year_offset = 21, # 21 is 2025
    years_ahead = 10,
    scenario = "Moderate warming",
    model = model,
    threshold = 0.5,
    feature_list = features,
    generate_features_fn
):
    """
    Simulate future data and predict biodiversity anomaly risk over future time under a climate scenario.
    """

    n_months = years_ahead * 12
    results = []

    for t in range(n_months):

        # Time structure
        year = start_year_offset + (t // 12)
        month = (t % 12) + 1

        # Scenario dynamics (allows trends)
        climate = {
            "temp_anom_z": scenario["temp_start"] + year * scenario.get("temp_trend", 0),
            "precip_anom_z": scenario["precip_start"] + year * scenario.get("precip_trend", 0)
        }

        # Generate full feature set
        features_dict = generate_features_fn(state, month, climate)

        X = pd.DataFrame([features_dict])[feature_list]

        # Predict probability
        proba = model.predict_proba(X)[0, 1]

        # Apply threshold
        pred = int(proba >= threshold)

        results.append({
            "year": year,
            "month": month,
            "probability": proba,
            "prediction": pred,
            "temp_anom_z": climate["temp_anom_z"],
            "precip_anom_z": climate["precip_anom_z"]
        })

    df = pd.DataFrame(results)

    # Summary stats
    summary = {
        "total_months": len(df),
        "anomaly_months": int(df["prediction"].sum()),
        "anomaly_rate": df["prediction"].mean(),
        "mean_probability": df["probability"].mean()
    }

    return df, summary

---
---

Forecasting  

Define future years and months

In [31]:
years_ahead = 10
months = years_ahead * 12

Generate future df

In [32]:
year2025_offset = 21

future_df = pd.DataFrame({
    "month": np.tile(np.arange(1,13), years_ahead),
    "year_offset": np.repeat(np.arange(year2025_offset, year2025_offset + years_ahead), 12)
})

future_df['month_sin'] = np.sin(2*np.pi * future_df['month'] / 12)
future_df['month_cos'] = np.cos(2*np.pi * future_df['month'] / 12)

future_df = future_df.drop(columns=['month'])

future_df.head()

,year_offset,month_sin,month_cos
0,21,0.500000,8.660254e-01
1,21,0.866025,5.000000e-01
2,21,1.000000,6.123234e-17
3,21,0.866025,-5.000000e-01
4,21,0.500000,-8.660254e-01


Define scenarios

In [33]:
SCENARIOS = {
    "Moderate warming": {
        "temp_start": 0.5,
        "temp_trend": 0.05,   # per year
        "precip_start": 0.0,
        "precip_trend": -0.02
    },
    "Severe warming": {
        "temp_start": 1.0,
        "temp_trend": 0.1,
        "precip_start": -0.5,
        "precip_trend": -0.05
    }
}

Apply climate scenarios - define trend over time for temp_anom_z and precip_anom_z

In [34]:
def scenario_trend(t, scenario):
    return {
        "temp_anom_z": scenario["temp_start"] + t * scenario["temp_trend"],
        "precip_anom_z": scenario["precip_start"] + t * scenario["precip_trend"]
    }

Generate features

In [35]:
baseline = df.groupby(['state','month']).mean()

TypeError: agg function failed [how->mean,dtype->object]

In [ ]:
def generate_features(state, month, scenario):

    base = baseline.loc[(state, month)]

    temp_z = scenario["temp_anom_z"]
    precip_z = scenario["precip_anom_z"]

    return {
        "temp_anom_z": temp_z,
        "precip_anom_z": precip_z,

        "n_hot_days": base["n_hot_days"] + temp_z * 2,
        "heavy_rain_days": base["heavy_rain_days"] + max(0, precip_z * 2),

        "drought_index": temp_z - precip_z,

        # simple assumptions for lags
        "temp_anom_z_lag1": temp_z,
        "temp_anom_z_roll3": temp_z,
        "precip_anom_z_lag1": precip_z,
        "precip_anom_z_roll3": precip_z,
        "drought_index_roll3": temp_z - precip_z,

        "n_hot_days_roll3": base["n_hot_days"] + temp_z * 2,
        "heavy_rain_days_roll3": base["heavy_rain_days"] + max(0, precip_z * 2),
    }

In [ ]:
predictions = []
scenario = "Moderate warming"

for t in range(months):
    year = t // 12
    month = (t % 12) + 1

    climate = scenario_trend(year, scenario)

    features = generate_features(state, month, climate)

    X = pd.DataFrame([features])[feature_list]

    proba = model.predict_proba(X)[0,1]
    pred = int(proba > threshold)

    predictions.append({
        "month": month,
        "year": year,
        "proba": proba,
        "pred": pred
    })

---
---

In [ ]:
st.title("Wildsignal: Biodiversity Anomaly Predictor")

# columns are missing: 
#  'temp_anom_z_roll3', 'precip_anom_z_roll3', 'log_n_obs', 'n_hot_days', 
#  'heavy_rain_days_roll3', 'n_hot_days_roll3', 'heavy_rain_days', 'temp_anom_z', 
#  'precip_anom_z', 'drought_index_roll3'

# Sidebar inputs
state = st.selectbox("State", df['state'].unique())
year = st.slider("Year", 2004, 2024)
month = st.slider("Month", 1, 12)
# Climate scenarios


# Example: generate features
input_data = pd.DataFrame({
    'state': [state],
    'year_offset': [year - 2004],
    'month_sin': [np.sin(2*np.pi*month/12)],
    'month_cos': [np.cos(2*np.pi*month/12)],
    # add other features here
})

# Predict
proba = model.predict_proba(input_data)[0,1]

st.write(f"Probability of biodiversity anomaly: {proba:.2f}")

2026-04-10 09:19:21.022 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 09:19:21.023 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 09:19:21.025 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 09:19:21.027 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 09:19:21.028 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 09:19:21.030 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 09:19:21.033 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-10 09:19:21.034 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

ValueError: columns are missing: {'temp_anom_z_roll3', 'precip_anom_z_roll3', 'log_n_obs', 'n_hot_days', 'heavy_rain_days_roll3', 'n_hot_days_roll3', 'heavy_rain_days', 'temp_anom_z', 'precip_anom_z', 'drought_index_roll3'}